In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'jax', 'jaxlib', 'flax', 'optax', 'orbax-checkpoint', 'tensorflow', 'protobuf', 'ml-dtypes', 'safetensors']
imports = {'pillow': 'PIL', 'orbax-checkpoint': 'orbax', 'protobuf': 'google.protobuf', 'ml-dtypes': 'ml_dtypes'}
pinned = {'jax': ('0.12.0.1', 'jax==0.12.0.1', 'jax[cuda12]==0.12.0.1', 'exact'), 'jaxlib': ('0.12.0.1', 'jaxlib==0.12.0.1', 'jaxlib==0.12.0.1', 'exact'), 'flax': ('0.12.12', 'flax==0.12.12', 'flax==0.12.12', 'exact'), 'optax': ('0.2.8', 'optax==0.2.8', 'optax==0.2.8', 'exact'), 'orbax-checkpoint': ('0.12.0', 'orbax-checkpoint==0.12.0', 'orbax-checkpoint==0.12.0', 'exact'), 'safetensors': ('0.8.0', 'safetensors==0.8.0', 'safetensors==0.8.0', 'exact')}
fallbacks = {'tensorflow': 'tensorflow==2.21.0', 'protobuf': 'protobuf==7.34.1', 'ml-dtypes': 'ml-dtypes==0.5.4'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'jax' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, _, _, match) in pinned.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "061a82116be00c585cab41d5b2a790e2a4f96d44"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/061a82116be00c585cab41d5b2a790e2a4f96d44/d2l"
for name in ('__init__.py', 'jax.py', 'nnx_resnet.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Saving, Loading, and Pretrained Weights

A trained model is a fragile thing: it lives only in the memory of the
process that trained it. When the process ends, because the script finished,
crashed, or ran on a cloud machine that was reclaimed, the weights are gone
unless they were written to disk. For the small models of earlier chapters
this is a nuisance: a few minutes of training to repeat. For a large model
it can mean weeks of computation, and at scale interruptions are the rule.
In a 54-day snapshot of the pre-training of Llama 3 405B on up to 16,384
GPUs, Meta recorded 419 unexpected interruptions, about one every three
hours [@Grattafiori.Dubey.Jauhri.ea.2024]. A run that cannot continue
from where it stopped would never finish.

Saved weights also let others use a model without training it: in practice,
many models start from weights that someone else trained, as the
fine-tuning examples of that section do. Both uses raise the
same questions: what must be written to disk, and how is it read back?

What has to be saved follows from that section. A model is code
plus state. The code defines the layers and the forward computation; the
*state* is the set of named tensors that training produced, parameters and
buffers alike. We save only the state, as a mapping from names to tensors,
and restore a model by building its structure from code and loading the
tensors into it. Resuming training needs more than the weights: it also
needs the optimizer's running averages, the random-number streams, and the
step counter, which makes the file several times larger.

This is a practical section, about files: what goes into them, which of
them can be trusted, and what a load must check. We first save and load the
state of a small network. We then ask which file formats are safe to load
from a stranger, write a checkpoint from which training resumes exactly,
and load pretrained weights into a network with a new output layer.

In [ ]:
import math
import numpy as np
import jax
from jax import numpy as jnp
from flax import nnx
import optax
from d2l import jax as d2l

In [ ]:
import json
import orbax.checkpoint as ocp
import os
import struct
from collections import Counter
from d2l.nnx_resnet import ResNet50
from dataclasses import asdict, dataclass
from flax import traverse_util
from safetensors.flax import load_file, save_file

## Model State

A model holds many tensors, so a file format for its state must store a
collection of named tensors. The framework builds that collection for us.
Its names are the paths through the module tree of
that section, and its values are the tensors. Here is the
state of an MLP with one hidden layer of 256 units.

`nnx.state` returns the variables as a tree whose paths follow the module
graph (`hidden.kernel`, `output.bias`; Flax calls a weight matrix a
*kernel*).

In [ ]:
class MLP(nnx.Module):
    def __init__(self, rngs):
        self.hidden = nnx.Linear(20, 256, rngs=rngs)
        self.output = nnx.Linear(256, 10, rngs=rngs)

    def __call__(self, x):
        return self.output(nnx.relu(self.hidden(x)))

net = MLP(nnx.Rngs(d2l.get_key()))
X = jax.random.normal(d2l.get_key(), (2, 20))
Y = net(X)
params = nnx.state(net, nnx.Param)
[(path, tuple(value.shape)) for path, value in params.flat_state()]

The state holds names, shapes, and values, and that is all. It does not
record the `MLP` class, its `forward` computation, or the activation
function. Loading it therefore needs code that builds a model with the same
names and shapes. Renaming an attribute, say `output` to `head`, changes the
names and breaks the match even though the computation stays the same. The
last part of this section shows how to detect such mismatches when loading.

## Pickle and safetensors

A file that stays on our own disk can be in any format. A file downloaded
from a model hub, or received from a colleague, raises a question that
tensors on our own disk do not: can loading it do anything besides creating
tensors? Unfortunately, the answer depends on the format.

### Code Execution on Load

Python's standard serialization, `pickle`, does not store data. It stores a
program that reconstructs the data, and loading the file runs that program.
The program may call any function the loading process can import. An object
tells pickle what to call through its `__reduce__` method, which returns a
function and its arguments. The class below asks for `print`; a malicious
file would ask for something like `os.system`.

In [ ]:
class Tripwire:
    def __reduce__(self):  # pickle calls print(...) to rebuild the object
        return (print, ('*** payload executed while loading ***',))

NumPy, whose formats JAX uses, writes pickle whenever it saves an object
that is not a plain array. Loading such a file with `allow_pickle=True` runs
the payload:

In [ ]:
np.save('tripwire.npy', np.array(Tripwire(), dtype=object))  # pickled
_ = np.load('tripwire.npy', allow_pickle=True)  # the payload runs here

NumPy's default, `allow_pickle=False`, refuses the file instead:

In [ ]:
try:
    np.load('tripwire.npy')  # allow_pickle=False, the default
except ValueError as e:
    print(type(e).__name__, e)

A refusing loader protects us only as long as its checks are complete, and
such checks have had holes, in PyTorch's `weights_only` loader among others.
A format that contains no program at all removes the question.


### The safetensors Format

The *safetensors* format contains no program: a file holds tensors and a
description of them, and nothing else. It has three parts, shown in
the figure: an 8-byte little-endian integer that
gives the length of a header, the header itself as JSON text, and the raw
bytes of all tensors back to back. The header names each tensor and records
its dtype, its shape, and the range of bytes it occupies after the header.
Nothing in the file is executed when it is loaded; a loader only has to check
that the header is well formed and that the ranges lie inside the file.
Because the ranges are known in advance, a reader can map the file into
memory and read one tensor without reading the others. The format does not
depend on any framework, and it is the default format for weights on the
Hugging Face Hub.

![The safetensors file of the MLP below as one strip of bytes: an 8-byte length, the JSON header with each tensor's dtype, shape, and byte range, and the raw bytes of the four tensors back to back, in the order of their names. The offsets under the data are the ranges that the header records.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/bg-safetensors-layout.svg)

We save the state of the MLP as a safetensors file, load it into a second
instance, and check that the two networks compute the same outputs.

A safetensors file stores a flat mapping from names to tensors, while the
NNX state is a nested tree. `traverse_util.flatten_dict` turns the tree into
a mapping from paths to arrays, and we join each path with dots;
`unflatten_dict` reverses this.

In [ ]:
def to_flat(tree):
    return {'.'.join(path): value
            for path, value in traverse_util.flatten_dict(tree).items()}

def from_flat(flat):
    return traverse_util.unflatten_dict(
        {tuple(name.split('.')): value for name, value in flat.items()})

save_file(to_flat(nnx.to_pure_dict(params)), 'mlp.safetensors')
clone = MLP(nnx.Rngs(d2l.get_key()))
clone_params = nnx.state(clone, nnx.Param)
nnx.replace_by_pure_dict(clone_params, from_flat(load_file('mlp.safetensors')))
nnx.update(clone, clone_params)
bool(jnp.array_equal(clone(X), Y))

The round trip is exact. Because the header is JSON at a known offset, we
can read it with nothing but the Python standard library:

In [ ]:
with open('mlp.safetensors', 'rb') as f:
    n = struct.unpack('<Q', f.read(8))[0]  # header length, little-endian
    header = json.loads(f.read(n))
print('header length:', n, 'bytes')
for name, entry in sorted(header.items(), key=lambda e: e[1]['data_offsets']):
    print(f"{name:18s} {entry['dtype']} {str(entry['shape']):10s} "
          f"bytes {entry['data_offsets']}")

Each byte range is the number of entries times 4 bytes, since the dtype `F32`
is float32. The bias of the hidden layer has 256 entries and occupies the
first 1,024 bytes of the data; the weight matrix of the hidden layer, with
$256 \times 20 = 5{,}120$ entries, takes the next 20,480 bytes. Every range
starts where the previous one ends, so the data contains nothing but the
tensors.

For files that stay on our own machine, the framework's native format serves
well. For files that cross a trust boundary, in either direction, we use a
format that contains no program and publish a checksum with the file so that
the recipient can detect a corrupted or replaced download.

## Training Checkpoints

A file of weights is enough to use a model, but not to continue training it.
The reason is that the optimizer may have a memory of its own. Many of the
later chapters train with *Adam* (that section), which keeps two
running averages for every parameter (that section):
$\mathbf{m}$, of its gradient $\mathbf{g}$, and $\mathbf{v}$, of the
gradient's elementwise square. Each parameter moves against its averaged
gradient, by an amount scaled down where its gradients have been large:

$$
\mathbf{w} \leftarrow \mathbf{w} - \eta\,
\frac{\hat{\mathbf{m}}}{\sqrt{\hat{\mathbf{v}}} + \epsilon}.
$$

Here $\eta$ is the learning rate, $\epsilon$ a small constant such as
$10^{-8}$, and the division and square root act elementwise. The hats mark a
rescaling that corrects for the start of both averages at zero; its formula
can wait for that section. What matters here is that $\mathbf{m}$ and
$\mathbf{v}$ carry the history of the run: drop them, and the optimizer
behaves as if training had just begun.

A checkpoint from which training can resume therefore holds five things
(the figure): the model state; the optimizer
state, here $\mathbf{m}$ and $\mathbf{v}$; the state of the random-number
generators, so that data shuffling and dropout continue the same streams;
the step counter, which Adam's correction uses and which also sets the
position in any learning-rate schedule; and the configuration that sizes the
model when it is rebuilt. Gradients are not among them, since the next step
recomputes them. This leaves 12 of the 16 bytes per parameter that training
with Adam in fp32 stores (that section): the weights and the two
averages. For a model with 8 billion parameters the checkpoint therefore
takes 96 GB, three times the 32 GB of its fp32 weights and six times the
16 GB file that is enough for inference in bf16 (that section).

![A checkpoint file's five compartments, each paired by an arrow with what it restores on resume: model state with the weights, optimizer state with Adam's two running averages, RNG state with the data order and dropout, step with the position in the schedule, and config with the architecture.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/bg-checkpoint-contents.svg)

A checkpoint must also survive a crash while it is being written. If the new
file replaces the old one in place, a crash in the middle leaves neither. We
therefore write to a temporary name and rename the finished file; a rename
within one file system is atomic, so a reader sees either the old checkpoint
or the new one.

In NNX the model and the optimizer expose their state as trees, and the
random-number key, the step, and the configuration are three more branches;
in JAX a key is an array like any other. Orbax, the JAX checkpointing
library, writes such a tree to a temporary directory and renames it when it
is complete. We overwrite one directory with `force=True` so that the
notebook can run twice; a real run writes numbered steps through
`CheckpointManager` and keeps the last complete one.

The experiment trains a small regressor: an MLP with one hidden layer of 64
units, fit by Adam with learning rate 0.05 to 256 examples of a noisy linear
function of 20 inputs. Each step computes the gradient on a minibatch of 64
examples drawn at random. The draws come from a random-number generator of
their own, seeded with 0, so the order of the minibatches is part of the
state that a checkpoint must save. The configuration is a dataclass, the
form used in that section; it is what a rebuild reads to
size the model, so it goes into the checkpoint.

In [ ]:
@dataclass
class Config:
    num_inputs: int = 20
    num_hiddens: int = 64
    num_outputs: int = 1
    batch_size: int = 64
    lr: float = 0.05

We train for 200 steps, save a checkpoint, and then let the same run go on
for 5 more steps, as if nothing had happened. These 205 uninterrupted steps
are the reference that a resumed run must match. The function `train`
returns the loss of each minibatch, computed before the step that uses it.

In [ ]:
def build(cfg, rngs):
    xavier = nnx.initializers.xavier_uniform()  # biases start at zero
    return nnx.Sequential(
        nnx.Linear(cfg.num_inputs, cfg.num_hiddens, kernel_init=xavier,
                   rngs=rngs),
        nnx.relu,
        nnx.Linear(cfg.num_hiddens, cfg.num_outputs, kernel_init=xavier,
                   rngs=rngs))

def fresh_state(cfg):
    model = build(cfg, nnx.Rngs(d2l.get_key()))
    return model, nnx.Optimizer(model, optax.adam(cfg.lr), wrt=nnx.Param)

def checkpoint_tree(model, optimizer, key, step, cfg):
    return {'model': nnx.to_pure_dict(nnx.state(model)),
            'optimizer': nnx.to_pure_dict(nnx.state(optimizer)),
            'rng': key, 'step': step, 'cfg': asdict(cfg)}

cfg = Config()
data = jax.random.normal(d2l.get_key(), (256, cfg.num_inputs))
target = (data @ jax.random.normal(d2l.get_key(), (cfg.num_inputs, 1))
          + 0.1 * jax.random.normal(d2l.get_key(), (256, 1)))

def loss(model, X, y):
    return jnp.mean((model(X) - y) ** 2)

@nnx.jit
def step(model, optimizer, idx):
    l, grads = nnx.value_and_grad(loss)(model, data[idx], target[idx])
    optimizer.update(model, grads)
    return l

def train(model, optimizer, key, num_steps):
    losses = []
    for _ in range(num_steps):
        key, subkey = jax.random.split(key)
        idx = jax.random.randint(subkey, (cfg.batch_size,), 0, len(data))
        losses.append(step(model, optimizer, idx))
    return key, [float(l) for l in losses]

net, opt = fresh_state(cfg)
key = jax.random.key(0)                     # draws the minibatches
key, _ = train(net, opt, key, 200)
path = os.path.abspath('run-jax')           # orbax wants an absolute path
ckptr = ocp.StandardCheckpointer()
ckptr.save(path, checkpoint_tree(net, opt, key, 200, cfg), force=True)
ckptr.wait_until_finished()
key, uninterrupted = train(net, opt, key, 5)

To resume, a new process builds the model and the optimizer from the saved
configuration, seeds its generator as at the start of the run, loads the
checkpoint, and reads from the saved step how many steps remain. We resume
twice and compare the final weights with those of the uninterrupted run:
once from the full checkpoint, and once without the generator's state, as a
script would that restores everything else but simply seeds its generator
again.

Without a template, Orbax returns the saved tree as it was written: arrays,
the key, and the plain numbers of the step and the configuration. We read
the configuration, build the model, and write the restored values into the
model and the optimizer.

In [ ]:
def load_into(obj, saved):
    state = nnx.state(obj)
    nnx.replace_by_pure_dict(state, saved)
    nnx.update(obj, state)

def resume(path, num_steps, restore_optimizer=True, restore_rng=True):
    ckpt = ckptr.restore(path)              # the saved tree, as it is
    model, opt = fresh_state(Config(**ckpt['cfg']))
    load_into(model, ckpt['model'])
    if restore_optimizer:
        load_into(opt, ckpt['optimizer'])
    key = jax.random.key(0)                 # seeded as at the start
    if restore_rng:
        key = ckpt['rng']
    _, losses = train(model, opt, key, num_steps - ckpt['step'])
    return model, losses

def max_diff(a, b):
    return max(float(jnp.abs(p - q).max()) for p, q in
               zip(jax.tree.leaves(nnx.state(a)), jax.tree.leaves(nnx.state(b))))

In [ ]:
for name, kwargs in (('full checkpoint', {}),
                     ('without the generator', {'restore_rng': False})):
    resumed, _ = resume(path, 205, **kwargs)
    print(f'{name:22s} max |difference| {max_diff(resumed, net):.3g}')

Resumed from the full checkpoint, the run ends with exactly the weights of
the run that was never interrupted: the largest difference is 0, bit for
bit. Without the generator's state, the resumed steps draw the first
minibatches of the run again instead of the next ones, and the weights
differ by roughly 0.005 to 0.02. The step counter told both resumed runs
where to stop. Bitwise equality also needs computations that give the same
result every time they run, a topic that that section takes up.

A missing optimizer state does more damage, and the loss shows it. We
resume once more with the saved weights and generator but a newly created
Adam, and print the minibatch losses of the five steps next to those of the
full resume.

In [ ]:
_, full = resume(path, 205)
_, fresh = resume(path, 205, restore_optimizer=False)
print('saved optimizer:', [round(l, 4) for l in full])
print('fresh optimizer:', [round(l, 4) for l in fresh])

With the saved optimizer state, training continues where it stopped, and
the minibatch losses stay at their level at the checkpoint, about 0.01 or
less. With a fresh Adam, a single step raises the loss by a factor of
several hundred or more, and four steps later it has not recovered: it is
still a hundred times higher or more.

How can one step of an optimizer do so much damage to a model that was
nearly done? the equation explains it. A fresh Adam starts with
$\mathbf{m}$ and $\mathbf{v}$ at zero. After one update each holds a small
fraction of $\mathbf{g}$ and of $\mathbf{g} \odot \mathbf{g}$, and the
correction divides exactly that fraction out, which leaves
$\hat{\mathbf{m}} = \mathbf{g}$ and
$\hat{\mathbf{v}} = \mathbf{g} \odot \mathbf{g}$. The step of each
parameter then has size $\eta\, |g_i| / (|g_i| + \epsilon)$, almost exactly
$\eta$. Every parameter with a nonzero gradient moves by the full learning
rate of 0.05, however small that gradient is. The saved $\mathbf{v}$, by
contrast, still holds the larger squared gradients of earlier steps, so
$\sqrt{\hat{\mathbf{v}}}$ is large compared with the current gradients,
and the steps stay small. The closer a run is to its minimum, the smaller
its gradients and the larger the damage of a step that ignores their size.
Weights alone do not resume a run.

## Pretrained Weights

We now turn to the second use of a saved state: starting from weights that
someone else trained. A network trained on a large dataset such as ImageNet,
1.2 million images in 1,000 classes, computes features that are useful for
other image tasks too. To reuse it for a task with 10 classes, we keep all
layers except the last and replace that one, the *head*, with a new layer of
10 outputs. Our example is ResNet-50, a convolutional network with
about 25 million parameters (23.5 million without its 1,000-class head),
trained on ImageNet. We have not met convolutional networks yet, and luckily
we do not need to: for loading weights, ResNet-50 is simply a model state of
a few hundred named tensors.

The book's NNX ResNet-50, `d2l.nnx_resnet.ResNet50`, downloads Microsoft's
ImageNet weights as a safetensors file from the Hugging Face Hub, at a fixed
revision, and maps them into the NNX model.

In [ ]:
net = ResNet50.from_pretrained()                  # ~100 MB once
net.fc = nnx.Linear(2048, 10, rngs=nnx.Rngs(d2l.get_key()))  # new head
net.fc.kernel.shape

Weights from elsewhere often do not fit our model exactly, and a loader that
fills what it can without saying what it skipped hides the mismatch. To see
what a careful load reports, we remove two groups of entries from the
pretrained state. The first is the head, which we mean to replace anyway.
The second is the last of the four stages into which ResNet-50 groups its
layers, and it stands in for an accident: a file that lost part of its
contents. We load what remains into a new, randomly initialized ResNet-50
with a 10-class head, and compute two sets of names. *Missing* names are
parameters of the model that the file did not fill; they keep their random
initialization. *Unexpected* names are entries of the file that have no
place in the model.

We flatten both state trees to mappings from paths to arrays, take from the
file every path that the model has, and compute the two sets ourselves.

In [ ]:
def flat_state(model):
    return traverse_util.flatten_dict(nnx.to_pure_dict(nnx.state(model)))

damaged = {path: v for path, v in flat_state(net).items()
           if path[0] != 'fc' and path[:2] != ('stages', 3)}
target = ResNet50(num_classes=10, rngs=nnx.Rngs(d2l.get_key()))
fresh = flat_state(target)
state = nnx.state(target)
nnx.replace_by_pure_dict(state, traverse_util.unflatten_dict(
    {**fresh, **{p: v for p, v in damaged.items() if p in fresh}}))
nnx.update(target, state)
print('missing by block:', dict(Counter(
    f'stages.{p[1]}' if p[0] == 'stages' else p[0]
    for p in sorted(set(fresh) - set(damaged)))))
print('unexpected:', sorted(set(damaged) - set(fresh)))

The two entries of the head, its weight and its bias, are expected: the head
is new and is meant to start random. The entries of the last stage are not.
Had we loaded without looking, a large part of the network would now compute
its features with random weights, and no error would have told us. The set
of unexpected names is empty here; a nonempty one usually means that a layer
was renamed, so that its weights found no place in the model. In practice,
we state in advance which names may be missing, the head here, and treat
every other entry of either set as an error.

Training on the new task then usually starts by *freezing* the pretrained
layers, by the means of that section, so that the optimizer
changes only the new head. Whether and how far to unfreeze them later is a
question for that section.

All of this assumed that the state lives in main memory, and the computation
with it. that section moves both to GPUs.

## Summary

A saved model is state, not code: a mapping from names to tensors that
becomes a model only when code rebuilds a structure with the same names and
shapes. Loading a pickle file runs a program, so files from elsewhere belong
in a format without one, such as safetensors, which stores a JSON header and
raw bytes. A checkpoint that resumes training adds the optimizer state, the
random-number state, the step, and the configuration, and is written to a
temporary name and renamed into place. Resumed from all of them, our run
ended with the weights of the uninterrupted run, bit for bit. Without the
state of the generator that draws the minibatches, the two runs drift
apart. Without the optimizer state, Adam's
first step moves every parameter by the full learning rate, and the loss of
a model near its minimum jumps by orders of magnitude. Pretrained weights are
loaded by name into a model with a new head; the names that the file did
not fill, and the file entries that found no place, must be read and
checked against what was expected.

## Exercises

1. **Reasons to checkpoint.** Name two reasons to checkpoint a run that
   never leaves the machine it trains on, and state for each which
   compartments of the figure it needs.
2. [code] **Safetensors header.** The header length of the MLP's
   safetensors file is the 8-byte little-endian integer at the start of the
   file (the figure).
    1. Predict, then measure, how that length changes when the hidden width
       is doubled and when a third layer is added. Explain which parts of
       the header depend on tensor sizes at all.
    1. Read the hidden layer's bias from the file using only the header and
       file seeks, without the safetensors library. Decode the bytes with
       the dtype and shape that the header records, and check that the
       result equals the tensor that `load_file` returns.
    1. Explain why a pickle-based checkpoint cannot offer the same
       single-tensor read without deserializing the whole object graph.

3. [code] **Partial loading.** Define `MLP2`, a copy of `MLP` that inserts
   a second hidden layer `hidden2` of width 256 before the output layer and
   renames the attribute `output` to `head`.
    1. Load `mlp.safetensors` into `MLP2` with a key-set merge like the one
       that loads the damaged ResNet-50 state. Report the missing and
       unexpected key sets, and explain each entry.
    1. Rename the file's keys to the names that `MLP2` uses before the
       merge. Confirm that only `hidden2` keeps its fresh initialization,
       and make the merge raise on any unexpected key and on any missing
       key outside `hidden2`.

    *Adapted from Matthew Inkawhich, [Saving and Loading Models](https://docs.pytorch.org/tutorials/beginner/saving_loading_models.html), "Warmstarting Model Using Parameters from a Different Model".*

4. [code] **Dtype round trip.** ● A `float16` number stores 5 exponent bits
   and 10 mantissa bits, against 8 and 23 in `float32`
   (that section). Cast the parameters of an `MLP` to `float16`,
   save them, and load them back into a `float32` `MLP`.
    1. Predict the largest relative error of a restored parameter from the
       mantissa width. Measure the maximum absolute difference and, over
       the nonzero parameters, the maximum relative difference between the
       restored and the original values. Explain any parameter whose
       relative error exceeds the prediction.
    1. Compare the outputs of the original and the restored `MLP` on `X`,
       and decide whether the round trip is acceptable for inference.
    1. Adam keeps a running average of squared gradients.
       Compute what `float16` stores for the square of a gradient entry of
       $10^{-4}$, and explain why a resumable checkpoint keeps the
       optimizer state in `float32`.
5. [extended] **Crash-safe manifest.** ●
    1. Build a utility that writes model state, optimizer state, and a JSON
       metadata file under a fresh step-numbered prefix, then publishes the
       three by renaming into place one manifest that records each file's
       name, size, and checksum.
    1. Simulate two failures: a crash after the three files are written but
       before the manifest rename, and the truncation of one file of a
       published checkpoint. Show that reloading resumes from the previous
       complete checkpoint in the first case and rejects the corrupt
       checkpoint in the second.
    1. A rename is atomic, so a reader sees either the old manifest or the
       new one. POSIX does not require the data of the renamed files to
       reach the disk before the rename does. `os.fsync` forces the
       buffered writes of a file, or of a directory's entries, to disk.
       State where the utility must call `os.fsync` so that a power loss
       leaves either the previous checkpoint or the new one intact, and
       explain each call.

[Discussions](https://d2l.discourse.group/t/36757)